# Hive adapter v2 -- gate evaluation (G1-G3)

Generates `decide()` outputs for an evaluation set with the base model and the two v2 LoRA adapters
(TRAINING_SPEC.md §9-§10). **Scoring is not done here**: download `eval_out.zip` and score it locally
with `fine_tune_v2/score_eval.py` (runtime parser, sandbox, figure checks need no GPU).

**Before you start:** GPU **T4 x2**, **Internet on**, and these inputs (**+ Add Input**):
1. **The training notebook's output** (Notebook output tab -> the v2 training notebook, the run that produced
   `v2-lora-full-lr0.0002-s0` / `-s1`). The cell below finds the two `best/` folders anywhere under `/kaggle/input`.
2. Only for `SET = 'locked'`: a **private** dataset holding `locked.jsonl` and its `MANIFEST.json` from
   `fine_tune_v2/locked/` (never pushed to GitHub). Its sha256 is checked against the manifest.

**Order:** run `SET = 'dev'` first (290 rows, a check that the harness works on real outputs), score it,
then `SET = 'locked'` once (928 rows, the gate decision).

Runs (arm:mode:seed), sampled as `decide()` samples. Each adapter seed is paired with a base run of the
same sampling seed; the gates (G1-G3) use only these.
- `dev`: GPU 0 `base:sampled:0`, `v1:sampled:0`; GPU 1 `s1:sampled:1`, and `v2old:sampled:1` (the
  adapter before the E4 fix) when its `v2-lora-full-lr0.0002-s1` folder is attached (~1 h)
- `locked`: GPU 0 `base:sampled:0`, `base:sampled:1`; GPU 1 `s0:sampled:0`, `s1:sampled:1`, `v1:sampled:0` (~3-4 h)

`v1` is today's production adapter (`adapter/` in the repo), reported for v2-vs-v1; the gates compare
v2 with `base`. Set `TAG` to the adapters' folder prefix (`v2.1` after the E4 retrain).


In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/arkapravopal04/Swarm_neural_nets.git'
REPO_DIR = '/kaggle/working/Swarm_neural_nets'

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'{cmd} failed: {r.stdout} {r.stderr}')
    return r

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    run(['git', 'clone', REPO_URL, REPO_DIR])
run(['git', '-C', REPO_DIR, 'fetch', 'origin', 'main'])
run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'])
os.chdir(REPO_DIR)
for p in (REPO_DIR, os.path.join(REPO_DIR, 'fine_tune_v2')):
    if p not in sys.path:
        sys.path.insert(0, p)
print('synced to:', run(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s']).stdout.strip())
for f in ('fine_tune_v2/generate_eval.py', 'fine_tune_v2/parallel.py', 'serving.py'):
    assert os.path.exists(f), f'push {f} first'

In [ ]:
!pip -q install transformers==5.22.2 peft==0.21.2 accelerate==1.15.0  # as in the adapters' TRAINING_RECORD

## Config

In [ ]:
import glob, hashlib, json, torch

SET = 'dev'            # 'dev' first, then 'locked' once
BASE = 'Qwen/Qwen3-4B'
REVISION = None        # S5: the revision the adapters were trained on (None = what training used)
BATCH_TOKENS = 16000   # (longest prompt + 400) x batch size; lower it if a run runs out of memory
LIMIT = None           # e.g. 20 for a quick try

TAG = 'v2.1'           # which trained adapters to evaluate (folder prefix: v2 or v2.1)

def find_adapter(seed):
    hits = glob.glob(f'/kaggle/input/**/{TAG}-lora-full-lr0.0002-s{seed}/best/adapter_config.json', recursive=True)
    hits = [h for h in hits if '/adapters_pack/' not in h] or hits
    assert len(hits) == 1, f'attach exactly one training output: found {len(hits)} {TAG}-lora-full-lr0.0002-s{seed}/best'
    return os.path.dirname(hits[0])

ADAPTERS = {'s0': find_adapter(0), 's1': find_adapter(1)}
PROFILES = {k: json.load(open(os.path.join(v, 'serving.json'))) for k, v in ADAPTERS.items()}
# v1: today's production adapter, from the repo, served under the same v2 chat contract (its
# training format; its raw-text production serving never stops, probe 4). Reported, not a candidate.
ADAPTERS['v1'] = os.path.join(REPO_DIR, 'adapter')
# Old v2 (before the E4 fix), dev only, for the record -- used when its folder is attached.
old = [h for h in glob.glob('/kaggle/input/**/v2-lora-full-lr0.0002-s1/best/adapter_config.json', recursive=True)
       if '/adapters_pack/' not in h]
if TAG != 'v2' and len(old) == 1:
    ADAPTERS['v2old'] = os.path.dirname(old[0])
print('arms:', sorted(ADAPTERS))
for k, v in ADAPTERS.items():
    print(k, v)

if SET == 'dev':
    ROWS = 'fine_tune_v2/data/dev.jsonl'
else:
    hits = glob.glob('/kaggle/input/**/locked.jsonl', recursive=True)
    assert len(hits) == 1, f'attach the private locked dataset (found {hits})'
    ROWS = hits[0]
    man = json.load(open(os.path.join(os.path.dirname(ROWS), 'MANIFEST.json')))
    sha = hashlib.sha256(open(ROWS, 'rb').read()).hexdigest()
    assert sha == man['files']['locked.jsonl'], 'locked.jsonl does not match its sealed manifest'
    print('locked set verified:', man['count'], 'rows')
first = json.loads(open(ROWS).readline())
n_rows = sum(1 for _ in open(ROWS))
print('rows:', ROWS, n_rows)
# The rows, the adapters and this tokenizer must share one prompt builder and chat template (S5, S6).
from transformers import AutoTokenizer
import serving
tpl = serving.chat_template_sha256(AutoTokenizer.from_pretrained(BASE, **({'revision': REVISION} if REVISION else {})))
for k, prof in PROFILES.items():
    assert prof['prompt_builder_hash'] == first['prompt_builder_hash'], f'{k}: adapter and rows differ in prompt builder'
    assert prof['chat_template_sha256'] == tpl == first['chat_template_sha256'], f'{k}: chat template differs'
print('prompt builder', first['prompt_builder_hash'], '| chat template', tpl[:16], '| matched')
OUT = f'/kaggle/working/eval_out/{SET}'
JOBS_RUNS = ([['base:sampled:0', 'v1:sampled:0'],
              ['s1:sampled:1'] + (['v2old:sampled:1'] if 'v2old' in ADAPTERS else [])] if SET == 'dev' else
             [['base:sampled:0', 'base:sampled:1'], ['s0:sampled:0', 's1:sampled:1', 'v1:sampled:0']])
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 2, 'set the accelerator to GPU T4 x2'

## Generate

In [ ]:
import parallel

def cmd(runs):
    c = [sys.executable, 'fine_tune_v2/generate_eval.py', '--rows', ROWS, '--out-dir', OUT, '--base', BASE,
         '--batch-tokens', str(BATCH_TOKENS)]
    for k, v in ADAPTERS.items():
        if any(r.startswith(k + ':') for r in runs):  # a base-only job loads no adapter
            c += ['--adapter', f'{k}={v}']
    for r in runs:
        c += ['--run', r]
    if REVISION:
        c += ['--revision', REVISION]
    if LIMIT:
        c += ['--limit', str(LIMIT)]
    return c

jobs = [dict(name=f'gpu{g}', gpu=g, cmd=cmd(runs)) for g, runs in enumerate(JOBS_RUNS)]
codes = parallel.run_jobs(jobs, ready_marker='serving: chat template')
assert all(c == 0 for c in codes.values()), f'a job failed -- read its [gpuN] lines above: {codes}'

## Check and download

In [ ]:
import shutil
for f in sorted(glob.glob(os.path.join(OUT, '*.jsonl'))):
    rows = [json.loads(l) for l in open(f)]
    eos = sum(r['eos'] for r in rows) / max(1, len(rows))
    print(f'{os.path.basename(f):24s} {len(rows):4d} rows  stopped {eos:.3f}  mean tokens '
          f'{sum(r["n_new"] for r in rows) / max(1, len(rows)):.0f}')
shutil.make_archive('/kaggle/working/eval_out', 'zip', '/kaggle/working/eval_out')
print('download /kaggle/working/eval_out.zip')